# 06 — Diagramas Espacio-Tiempo y Estimación de Velocidad Eritrocitaria

Siguiendo Ellis et al. (1992) y MicroTools (Hilty et al., 2019).
Métodos: Transformada de Hough, correlación cruzada 2D.

In [ ]:
import os, cv2, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import correlate2d
from skimage.transform import hough_line, hough_line_peaks
from skimage.morphology import skeletonize

DATA_PATH = Path(os.environ.get("MICROCIRCULATION_DATA", "/content/drive/MyDrive/microcirculation"))
SEG_DIR   = DATA_PATH / "segmented"
ST_DIR    = DATA_PATH / "spacetime"
ST_DIR.mkdir(exist_ok=True)

CALIB_UM_PX = 1.0    # µm/píxel — ajustar
FPS         = 25.0   # frames/segundo — ajustar según video


## 6.1 Extraer línea central de un vaso

In [ ]:
def extract_centerline_points(skeleton: np.ndarray) -> list[tuple]:
    """Retorna lista de (row, col) de píxeles del esqueleto ordenados a lo largo del vaso."""
    pts = list(zip(*np.where(skeleton > 0)))
    if len(pts) < 2:
        return pts
    # Ordenar por distancia acumulada (nearest neighbor a lo largo del eje)
    ordered = [pts[0]]
    remaining = set(range(1, len(pts)))
    while remaining:
        last = ordered[-1]
        nearest = min(remaining,
                      key=lambda i: (pts[i][0]-last[0])**2 + (pts[i][1]-last[1])**2)
        ordered.append(pts[nearest])
        remaining.remove(nearest)
    return ordered


## 6.2 Construir diagrama espacio-tiempo

In [ ]:
def build_spacetime_diagram(frames_gray: list[np.ndarray],
                             centerline: list[tuple],
                             width: int = 3) -> np.ndarray:
    """
    Para cada frame, muestrea intensidad media a lo largo de la línea central
    con ancho de 'width' píxeles.
    Retorna matriz (n_frames x n_puntos_centreline).
    """
    n_pts = len(centerline)
    diagram = np.zeros((len(frames_gray), n_pts), dtype=np.float32)

    for t, frame in enumerate(frames_gray):
        h, w = frame.shape
        for j, (r, c) in enumerate(centerline):
            r0, r1 = max(0, r - width//2), min(h, r + width//2 + 1)
            c0, c1 = max(0, c - width//2), min(w, c + width//2 + 1)
            diagram[t, j] = frame[r0:r1, c0:c1].mean()

    return diagram


def visualize_spacetime(diagram: np.ndarray, fps: float,
                        calib: float, title: str = "") -> None:
    plt.figure(figsize=(10, 5))
    plt.imshow(diagram, aspect='auto', cmap='gray',
               extent=[0, diagram.shape[1] * calib,
                       diagram.shape[0] / fps, 0])
    plt.xlabel("Posición (µm)"); plt.ylabel("Tiempo (s)")
    plt.title(f"Diagrama espacio-tiempo — {title}")
    plt.colorbar(label="Intensidad")
    plt.tight_layout(); plt.show()


## 6.3 Estimar velocidad — Transformada de Hough

In [ ]:
def estimate_velocity_hough(diagram: np.ndarray,
                             fps: float, calib_um_px: float) -> float:
    """
    Detecta la pendiente dominante en el diagrama espacio-tiempo.
    Velocidad = (Δposición_µm / Δtiempo_s).
    """
    # Preprocesar para realzar bandas diagonales
    diag_norm = cv2.normalize(diagram, None, 0, 255,
                              cv2.NORM_MINMAX).astype(np.uint8)
    edges = cv2.Canny(diag_norm, 30, 80)

    # Hough con resolución angular fina
    tested_angles = np.linspace(-np.pi/2, np.pi/2, 360, endpoint=False)
    h, theta, d   = hough_line(edges, theta=tested_angles)
    _, angles, _  = hough_line_peaks(h, theta, d, num_peaks=5,
                                      threshold=0.4 * h.max())

    if len(angles) == 0:
        return np.nan

    # Ángulo principal → pendiente en px/frame → velocidad en µm/s
    angle_median = np.median(angles)
    # tan(90° - angle) = Δcol/Δrow = (px_espacio / px_tiempo)
    if abs(np.cos(angle_median)) < 1e-6:
        return np.nan

    slope_px_per_frame = np.tan(np.pi/2 - abs(angle_median))
    velocity_um_s      = slope_px_per_frame * calib_um_px * fps
    return float(velocity_um_s)


## 6.4 Estimar velocidad — Correlación cruzada 2D

In [ ]:
def estimate_velocity_xcorr(diagram: np.ndarray,
                              fps: float, calib_um_px: float,
                              lag_frames: int = 5) -> float:
    """
    Correlaciona franjas temporales desplazadas para detectar desplazamiento.
    """
    if diagram.shape[0] < lag_frames * 2:
        return np.nan

    ref  = diagram[:lag_frames].mean(axis=0)
    query= diagram[lag_frames:lag_frames*2].mean(axis=0)

    corr = np.correlate(ref - ref.mean(), query - query.mean(), mode='full')
    shift_px = int(np.argmax(corr) - (len(corr) // 2))

    velocity_um_s = (shift_px * calib_um_px * fps) / lag_frames
    return float(abs(velocity_um_s))


## 6.5 Pipeline completo por video

In [ ]:
def analyze_video_spacetime(video_path: str,
                             seg_dir: Path,
                             out_dir: Path,
                             fps: float = FPS,
                             calib: float = CALIB_UM_PX,
                             max_frames: int = 150) -> pd.DataFrame:
    """
    Lee frames del video, carga esqueleto de seg_dir,
    construye diagramas espacio-tiempo por vaso y estima velocidad.
    """
    cap     = cv2.VideoCapture(video_path)
    frames  = []
    fidx    = 0
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frames.append(gray)
        fidx += 1
    cap.release()

    # Cargar máscara segmentada del primer frame como proxy del esqueleto
    mask_path = seg_dir / "frame_00000_overlay.png"
    if not mask_path.exists():
        return pd.DataFrame()

    # Usar esqueleto de la máscara mediana
    all_masks = list(seg_dir.glob("*_overlay.png"))
    if not all_masks:
        return pd.DataFrame()

    skel_accum = np.zeros(frames[0].shape, dtype=np.float32)
    for mp in all_masks[:10]:
        overlay = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
        if overlay is not None and overlay.shape == skel_accum.shape:
            skel_accum += (overlay > 100).astype(np.float32)
    skeleton = (skel_accum > 5).astype(np.uint8)

    centerline = extract_centerline_points(skeleton)
    if len(centerline) < 10:
        return pd.DataFrame()

    diagram = build_spacetime_diagram(frames, centerline)
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(str(out_dir / "spacetime.npy"), diagram)

    vel_hough = estimate_velocity_hough(diagram, fps, calib)
    vel_xcorr = estimate_velocity_xcorr(diagram, fps, calib)

    return pd.DataFrame([{
        "video"       : Path(video_path).stem,
        "vel_hough"   : vel_hough,
        "vel_xcorr"   : vel_xcorr,
        "vel_mean"    : np.nanmean([vel_hough, vel_xcorr]),
        "n_frames"    : len(frames),
        "cl_length_px": len(centerline)
    }])


# Ejecutar
video_files = sorted(DATA_PATH.glob("*.mp4")) + sorted(DATA_PATH.glob("*.avi"))
all_vel     = []

for vf in video_files:
    seg_subdir = SEG_DIR / vf.stem
    out_subdir = ST_DIR  / vf.stem
    result     = analyze_video_spacetime(str(vf), seg_subdir, out_subdir)
    if not result.empty:
        all_vel.append(result)
        print(f"{vf.stem}: Hough={result['vel_hough'].iloc[0]:.1f}  "
              f"Xcorr={result['vel_xcorr'].iloc[0]:.1f} µm/s")

vel_df = pd.concat(all_vel, ignore_index=True) if all_vel else pd.DataFrame()
vel_df.to_csv(DATA_PATH / "velocity_estimates.csv", index=False)
print(f"\nVelocidades estimadas: {len(vel_df)} videos")
